#### Load the dataset

In [1]:
from sklearn.datasets import fetch_kddcup99
import pandas as pd

kdd = fetch_kddcup99(percent10 = True)

df = pd.DataFrame(
    kdd.data,
    columns = kdd.feature_names
)

df['target'] = kdd.target

df.sample(7)

,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,target
188398,0,b'icmp',b'ecr_i',b'SF',1032,0,0,0,0,0,...,255,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,b'smurf.'
240814,0,b'icmp',b'ecr_i',b'SF',1032,0,0,0,0,0,...,255,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,b'smurf.'
353888,0,b'tcp',b'private',b'S0',0,0,0,0,0,0,...,18,0.07,0.07,0.0,0.0,1.0,1.0,0.0,0.0,b'neptune.'
411835,0,b'icmp',b'ecr_i',b'SF',520,0,0,0,0,0,...,255,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,b'smurf.'
312442,0,b'icmp',b'ecr_i',b'SF',1032,0,0,0,0,0,...,255,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,b'smurf.'
288266,0,b'icmp',b'ecr_i',b'SF',1032,0,0,0,0,0,...,255,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,b'smurf.'
40104,0,b'tcp',b'http',b'SF',54540,8314,0,0,0,2,...,255,1.0,0.0,0.0,0.0,0.0,0.0,0.06,0.06,b'back.'


In [2]:
df['target'].value_counts()

target
b'smurf.'              280790
b'neptune.'            107201
b'normal.'              97278
b'back.'                 2203
b'satan.'                1589
b'ipsweep.'              1247
b'portsweep.'            1040
b'warezclient.'          1020
b'teardrop.'              979
b'pod.'                   264
b'nmap.'                  231
b'guess_passwd.'           53
b'buffer_overflow.'        30
b'land.'                   21
b'warezmaster.'            20
b'imap.'                   12
b'rootkit.'                10
b'loadmodule.'              9
b'ftp_write.'               8
b'multihop.'                7
b'phf.'                     4
b'perl.'                    3
b'spy.'                     2
Name: count, dtype: int64

In [3]:
df.shape

(494021, 42)

#### Preprocessing

In [4]:
print("Missing values : \n")
df.isnull().sum()

Missing values : 



duration                       0
protocol_type                  0
service                        0
flag                           0
src_bytes                      0
dst_bytes                      0
land                           0
wrong_fragment                 0
urgent                         0
hot                            0
num_failed_logins              0
logged_in                      0
num_compromised                0
root_shell                     0
su_attempted                   0
num_root                       0
num_file_creations             0
num_shells                     0
num_access_files               0
num_outbound_cmds              0
is_host_login                  0
is_guest_login                 0
count                          0
srv_count                      0
serror_rate                    0
srv_serror_rate                0
rerror_rate                    0
srv_rerror_rate                0
same_srv_rate                  0
diff_srv_rate                  0
srv_diff_h

In [5]:
dup = df.duplicated().sum()
print(f"Duplicate values :\n{dup}")

Duplicate values :
348435


In [6]:
df.drop_duplicates(inplace = True)
print("Shape of the dataset after removing duplicates : ", df.shape)

Shape of the dataset after removing duplicates :  (145586, 42)


Converting bytes to string

In [7]:
for col in df.select_dtypes(include = ['object', 'string']).columns:
    try:
        df[col] = df[col].str.decode('utf-8')
    except:
        pass

Standardizing the text

In [8]:
for col in df.select_dtypes(include = ['object', 'string']).columns:
    df[col] = df[col].astype(str).str.lower().str.strip()

Label formatting

In [9]:
df['target'] = df['target'].astype(str).str.replace('.', '', regex = False)

Reducing noise in `service` column

In [10]:
count_service = df['service'].value_counts()
print(count_service)
rare = count_service[count_service < 200].index
df['service'] = df['service'].replace(rare, 'others')

service
http        62054
private     49057
smtp         9721
domain_u     5425
other        4769
            ...  
x11            11
tim_i           5
pm_dump         1
tftp_u          1
red_i           1
Name: count, Length: 66, dtype: int64


Dropping constant columns

In [11]:
const = [c for c in df.columns if df[c].nunique() == 1]

df.drop(const, axis = 1, inplace = True)
print("Removed columns that are constant are :", const, "\n")
print(df.shape)

Removed columns that are constant are : ['num_outbound_cmds', 'is_host_login'] 

(145586, 40)


In [12]:
df['binary_target'] = df['target'].apply(lambda x: 0 if x == 'normal' else 1)
df['binary_target'].value_counts()

binary_target
0    87832
1    57754
Name: count, dtype: int64

In [13]:
X = df.drop(['target', 'binary_target'], axis = 1)
y = df['binary_target']

In [14]:
X = pd.get_dummies(X, columns = ['protocol_type', 'service', 'flag'])

In [15]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size = 0.2, random_state = 45
)

In [16]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [17]:
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

X_train shape: (116468, 66)
X_test shape: (29118, 66)


In [18]:
from sklearn.decomposition import PCA

pca = PCA(n_components = 0.95)
X_train = pca.fit_transform(X_train)
X_test = pca.transform(X_test)

#### ISOLATION FOREST

In [19]:
X_train = X_train[y_train == 0]

In [54]:
from sklearn.ensemble import IsolationForest

IF_model = IsolationForest(contamination = 0.2, random_state = 45)
IF_model.fit(X_train)

,"n_estimators n_estimators: int, default=100The number of base estimators in the ensemble.",100
,"max_samples max_samples: ""auto"", int or float, default=""auto""The number of samples to draw from X to train each base estimator.- If int, then draw `max_samples` samples.- If float, then draw `max_samples * X.shape[0]` samples.- If ""auto"", then `max_samples=min(256, n_samples)`.If max_samples is larger than the number of samples provided,all samples will be used for all trees (no sampling).",'auto'
,"contamination contamination: 'auto' or float, default='auto'The amount of contamination of the data set, i.e. the proportionof outliers in the data set. Used when fitting to define the thresholdon the scores of the samples.- If 'auto', the threshold is determined as in the original paper.- If float, the contamination should be in the range (0, 0.5]... versionchanged:: 0.22 The default value of ``contamination`` changed from 0.1 to ``'auto'``.",0.2
,"max_features max_features: int or float, default=1.0The number of features to draw from X to train each base estimator.- If int, then draw `max_features` features.- If float, then draw `max(1, int(max_features * n_features_in_))` features.Note: using a float number less than 1.0 or integer less than number offeatures will enable feature subsampling and leads to a longer runtime.",1.0
,"bootstrap bootstrap: bool, default=FalseIf True, individual trees are fit on random subsets of the trainingdata sampled with replacement. If False, sampling without replacementis performed.",False
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for :meth:`fit`. ``None`` means 1unless in a :obj:`joblib.parallel_backend` context. ``-1`` means usingall processors. See :term:`Glossary ` for more details.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseudo-randomness of the selection of the featureand split values for each branching step and each tree in the forest.Pass an int for reproducible results across multiple function calls.See :term:`Glossary `.",45
,"verbose verbose: int, default=0Controls the verbosity of the tree building process.",0
,"warm_start warm_start: bool, default=FalseWhen set to ``True``, reuse the solution of the previous call to fitand add more estimators to the ensemble, otherwise, just fit a wholenew forest. See :term:`the Glossary `... versionadded:: 0.21",False


In [55]:
IF_y_pred = IF_model.predict(X_test)

IF_y_pred = [0 if pred == 1 else 1 for pred in IF_y_pred]
print(IF_y_pred[:100])

[0, 0, 1, 0, 1, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 1, 0, 1, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0]


In [56]:
from sklearn.metrics import classification_report, accuracy_score

IF_acc = accuracy_score(y_test, IF_y_pred)
print("Accuracy is :", IF_acc)

IF_report = classification_report(y_test, IF_y_pred)
print(IF_report)

Accuracy is : 0.6118895528539048
              precision    recall  f1-score   support

           0       0.64      0.80      0.71     17495
           1       0.52      0.33      0.40     11623

    accuracy                           0.61     29118
   macro avg       0.58      0.56      0.56     29118
weighted avg       0.59      0.61      0.59     29118



#### One Class SVM

In [23]:
from sklearn.svm import OneClassSVM

SVM_model = OneClassSVM(
    kernel = 'rbf',
    gamma = 'scale',
    nu = 0.05
)

SVM_model.fit(X_train)

,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm.If none is given, 'rbf' will be used. If a callable is given it isused to precompute the kernel matrix.",'rbf'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"nu nu: float, default=0.5An upper bound on the fraction of trainingerrors and a lower bound of the fraction of supportvectors. Should be in the interval (0, 1]. By default 0.5will be taken.",0.05
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide `.",True
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"verbose verbose: bool, default=FalseEnable verbose output. Note that this setting takes advantage of aper-process runtime setting in libsvm that, if enabled, may not workproperly in a multithreaded context.",False
,"max_iter max_iter: int, default=-1Hard limit on iterations within solver, or -1 for no limit.",-1


In [24]:
SVM_y_pred = SVM_model.predict(X_test)
SVM_y_pred = [0 if pred == 1 else 1 for pred in SVM_y_pred]

acc = accuracy_score(y_test, SVM_y_pred)
print("Accuracy is :", acc)

report = classification_report(y_test, SVM_y_pred)
print(report)

Accuracy is : 0.9607802733704238
              precision    recall  f1-score   support

           0       0.99      0.95      0.97     17495
           1       0.93      0.98      0.95     11623

    accuracy                           0.96     29118
   macro avg       0.96      0.96      0.96     29118
weighted avg       0.96      0.96      0.96     29118



#### AutoEncoder

In [25]:
from sklearn.neural_network import MLPRegressor

autoencoder = MLPRegressor(
    hidden_layer_sizes = (32,16,32),
    activation = 'relu',
    solver = 'adam',
    max_iter = 100,
    random_state = 45
)

autoencoder.fit(X_train, X_train)

,"loss loss: {'squared_error', 'poisson'}, default='squared_error'The loss function to use when training the weights. Note that the""squared error"" and ""poisson"" losses actually implement""half squares error"" and ""half poisson deviance"" to simplify thecomputation of the gradient. Furthermore, the ""poisson"" loss internally usesa log-link (exponential as the output activation function) and requires``y >= 0``... versionchanged:: 1.7 Added parameter `loss` and option 'poisson'.",'squared_error'
,"hidden_layer_sizes hidden_layer_sizes: array-like of shape(n_layers - 2,), default=(100,)The ith element represents the number of neurons in the ithhidden layer.","(32, ...)"
,"activation activation: {'identity', 'logistic', 'tanh', 'relu'}, default='relu'Activation function for the hidden layer.- 'identity', no-op activation, useful to implement linear bottleneck, returns f(x) = x- 'logistic', the logistic sigmoid function, returns f(x) = 1 / (1 + exp(-x)).- 'tanh', the hyperbolic tan function, returns f(x) = tanh(x).- 'relu', the rectified linear unit function, returns f(x) = max(0, x)",'relu'
,"solver solver: {'lbfgs', 'sgd', 'adam'}, default='adam'The solver for weight optimization.- 'lbfgs' is an optimizer in the family of quasi-Newton methods.- 'sgd' refers to stochastic gradient descent.- 'adam' refers to a stochastic gradient-based optimizer proposed by Kingma, Diederik, and Jimmy BaFor a comparison between Adam optimizer and SGD, see:ref:`sphx_glr_auto_examples_neural_networks_plot_mlp_training_curves.py`.Note: The default solver 'adam' works pretty well on relativelylarge datasets (with thousands of training samples or more) in terms ofboth training time and validation score.For small datasets, however, 'lbfgs' can converge faster and performbetter.",'adam'
,"alpha alpha: float, default=0.0001Strength of the L2 regularization term. The L2 regularization termis divided by the sample size when added to the loss.",0.0001
,"batch_size batch_size: int, default='auto'Size of minibatches for stochastic optimizers.If the solver is 'lbfgs', the regressor will not use minibatch.When set to ""auto"", `batch_size=min(200, n_samples)`.",'auto'
,"learning_rate learning_rate: {'constant', 'invscaling', 'adaptive'}, default='constant'Learning rate schedule for weight updates.- 'constant' is a constant learning rate given by 'learning_rate_init'.- 'invscaling' gradually decreases the learning rate ``learning_rate_`` at each time step 't' using an inverse scaling exponent of 'power_t'. effective_learning_rate = learning_rate_init / pow(t, power_t)- 'adaptive' keeps the learning rate constant to 'learning_rate_init' as long as training loss keeps decreasing. Each time two consecutive epochs fail to decrease training loss by at least tol, or fail to increase validation score by at least tol if 'early_stopping' is on, the current learning rate is divided by 5.Only used when solver='sgd'.",'constant'
,"learning_rate_init learning_rate_init: float, default=0.001The initial learning rate used. It controls the step-sizein updating the weights. Only used when solver='sgd' or 'adam'.",0.001
,"power_t power_t: float, default=0.5The exponent for inverse scaling learning rate.It is used in updating effective learning rate when the learning_rateis set to 'invscaling'. Only used when solver='sgd'.",0.5
,"max_iter max_iter: int, default=200Maximum number of iterations. The solver iterates until convergence(determined by 'tol') or this number of iterations. For stochasticsolvers ('sgd', 'adam'), note that this determines the number of epochs(how many times each data point will be used), not the number ofgradient steps.",100
,"shuffle shuffle: bool, default=TrueWhether to shuffle samples in each iteration. Only used whensolver='sgd' or 'adam'.",True


In [26]:
import numpy as np

X_reconstruct = autoencoder.predict(X_test)

error = np.mean(np.power(X_test - X_reconstruct, 2), axis = 1)

threshold = np.percentile(error, 60)
print(threshold)

0.046444552853542584


In [27]:
AE_pred = autoencoder.predict(X_test)
AE_pred = [1 if e > threshold else 0 for e in error]
print(AE_pred[:100])

[0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 1, 0, 1, 0, 1, 1, 1, 0, 0, 1, 0, 1, 1, 0, 0, 0, 1, 0, 0, 1, 1, 0, 1, 1, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 1, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0]


In [28]:
from sklearn.metrics import accuracy_score, classification_report

acc = accuracy_score(y_test, AE_pred)
print(acc)

report = classification_report(y_test, AE_pred)
print(report)

0.9769901778968336
              precision    recall  f1-score   support

           0       0.98      0.98      0.98     17495
           1       0.97      0.97      0.97     11623

    accuracy                           0.98     29118
   macro avg       0.98      0.98      0.98     29118
weighted avg       0.98      0.98      0.98     29118



### TESTING

In [57]:
import joblib

joblib.dump(rare, 'rare_services.pkl')
joblib.dump(const, 'constant_columns.pkl')
joblib.dump(list(X.columns), 'feature_columns.pkl')

joblib.dump(scaler, 'scaler.pkl')
joblib.dump(pca, 'pca.pkl')
joblib.dump(threshold, 'threshold.pkl')

joblib.dump(IF_model, 'IF_model.pkl')
joblib.dump(autoencoder, 'autoencoder.pkl')
joblib.dump(SVM_model, 'SVM_model.pkl')

['SVM_model.pkl']

In [58]:
rare = joblib.load('rare_services.pkl')
const = joblib.load('constant_columns.pkl')
columns = joblib.load('feature_columns.pkl')

scaler = joblib.load('scaler.pkl')
pca = joblib.load('pca.pkl')
threshold = joblib.load('threshold.pkl')

IF_model = joblib.load('IF_model.pkl')
autoencoder = joblib.load('autoencoder.pkl')
SVM_model = joblib.load('SVM_model.pkl')

In [59]:
def preprocessing(data):
    
    df = data.copy()

    for col in df.select_dtypes(include = ['object', 'string']).columns:
        df[col] = df[col].astype(str).str.lower().str.strip()

    df['service'] = df['service'].apply(lambda x: 'others' if x in rare else x)

    df.drop(columns = [c for c in const if c in df.columns], inplace = True)

    df = pd.get_dummies(df, columns = ['protocol_type', 'service', 'flag'])

    for c in columns:
        if c not in df.columns:
            df[c] = 0
    df = df[columns]

    df = scaler.transform(df)
    df = pca.transform(df)

    return df

In [60]:
import numpy as np
import pandas as pd

In [65]:
def predict(data):

    trained = preprocessing(data)

    isolation_pred = IF_model.predict(trained)
    isolation_pred = ['Normal' if pred == 1 else 'Anomaly' for pred in isolation_pred]

    svm_pred = SVM_model.predict(trained)
    svm_pred = ['Normal' if pred == 1 else 'Anomaly' for pred in svm_pred]

    reconstruct = autoencoder.predict(trained)
    err = np.mean((trained - reconstruct) ** 2, axis = 1)
    autoencoder_pred = ['Normal' if e < threshold else 'Anomaly' for e in err]

    predictions = pd.DataFrame({
        'ISOLATION FOREST' : isolation_pred,
        'ONE CLASS SVM' : svm_pred,
        'AUTOENCODER' : autoencoder_pred
    })

    return predictions

In [66]:
test_data = df.sample(7, random_state = 45)
test_data = test_data.drop(columns = ['target', 'binary_target'])

op = predict(test_data)
op

,ISOLATION FOREST,ONE CLASS SVM,AUTOENCODER
0,Normal,Normal,Normal
1,Normal,Normal,Normal
2,Anomaly,Anomaly,Anomaly
3,Normal,Anomaly,Anomaly
4,Anomaly,Anomaly,Anomaly
5,Normal,Normal,Normal
6,Normal,Normal,Normal
